# Horizon Uncertainty Analysis for WASDE Reports

Since the USDA already incorporates private data that I do not have access to, a predictive model with a higher point accuracy and significantly less input signal is not feasible.

A useful deliverable that can be extracted from the data that I have access to is concretely quantifying how wrong WASDE reports tend to be at each horizon, and when

# **Workflow**
1. Define the error/revision target

*Decide what 'truth' means: final NASS Crop Production Annual Summary, or the last WASDE report of the marketing year. Compute error = WASDE_estimate(commodity, market_year, report_month) minus truth, both as raw units and as a percent of truth so commodities are comparable. Keep the signed error (bias) and absolute error as separate columns. Both will be useful.*

2. Structure horizon explicitly

*Add a horizon variable — report sequence number within the marketing year (1st estimate, 2nd, ... final) rather than calendar month, since marketing years start at different calendar points by commodity. This is the axis the whole analysis pivots on, so get it right before anything else.*

3. Exploratory pass: error by horizon x commodity

*Build the core summary table/heatmap: mean bias and MAE/RMSE for every (commodity, horizon) cell. This alone answers 'does uncertainty shrink monotonically as the marketing year progresses' and 'which commodities are structurally noisier' — likely the headline result.*

4. Check for regime effects before pooling years

*Look at whether shock years (drought, trade disruptions) dominate the variance. Plot error by market year within a horizon to see if a few outlier years are driving the average, and decide whether to report robust stats (median/IQR) alongside means, or flag shock years separately rather than smoothing over them.*

5. Model the uncertainty itself

*Fit a model where the target is |error| or a quantile of error, with horizon and commodity as features — quantile regression or LightGBM's quantile objective works well here. This gives empirical prediction intervals (e.g. P10–P90 bands) you can attach to any current WASDE point estimate. The leave-one-market-year-out CV scheme still applies for validating these interval models.*

6. Validate interval calibration

*For any fitted quantile/interval model, check calibration on held-out years: does the P10–P90 band actually contain the true value ~80% of the time? This is the equivalent of the old accuracy metric, just applied to intervals instead of points.*

7. Package the deliverable

*Decide the output form — a lookup table (commodity, horizon) → expected error/interval, or a small function that takes a live WASDE estimate and horizon and returns a confidence band. This is what turns the analysis into something Darryl or downstream users can actually apply.*
